In [1]:
import pandas as pd
import re
import time

from src.services.container import Container
from src.services.scryfall_data_service import ScryfallDataService, BulkDataType
from src.services import image_service, file_service

container: Container = Container.get_singleton()

In [3]:
def get_image_from_card_list(cards_fpath: str, sds: ScryfallDataService) -> None:
    cards = re.findall(r"[0-9]+\s+([a-zA-Z\s,]+)\r\n", file_service.read_file(cards_fpath))
    cards_df: pd.DataFrame = sds.get_cards_from_bulk_data()
    oracle_ids = cards_df[cards_df.name.isin(cards)].oracle_id.tolist()
    image_service.save_images_as_grid(
        images=sds.get_card_images(oracle_ids).values(),
        output_path=f"./data/search_results/{int(time.time())}.png",
    )

get_image_from_card_list(
    cards_fpath="./data/lists/phils_commanders.txt",
    sds=container.get_service("scryfall_data")
)

In [25]:

cards_df: pd.DataFrame = container.get_scryfall_data_service().get_cards_from_bulk_data()
search_df: pd.DataFrame = cards_df[cards_df.name.str.contains("Lorehold, the Historian")]
for r in search_df[['name', 'set', 'mana_cost', 'power', 'toughness', 'oracle_text']].iterrows():
    print(f'{r[1]["name"]} ({r[1]["set"]}): {r[1]["mana_cost"]} - {r[1]["power"]}/{r[1]["toughness"]}')
    print(f'{r[1]["oracle_text"]}'.replace("\n", " // "))

Lorehold, the Historian (sos): {3}{R}{W} - 5/5
Flying, haste // Each instant and sorcery card in your hand has miracle {2}. (You may cast a card for its miracle cost when you draw it if it's the first card you drew this turn.) // At the beginning of each opponent's upkeep, you may discard a card. If you do, draw a card.
